In [1]:
import pandas as pd
import numpy as np

preds = pd.read_csv("dSets/test_predictions.csv")
X_test = pd.read_csv("dSets/X_test_FINAL.csv")
preds["order_value"] = X_test["order_value"].values

y_true = preds["y_true"]
probs = preds["log_reg_prob"]
order_value = preds["order_value"]

FLAG_COST = 40  

def fn_cost(order_value):
    return 150 + 50 + 0.02 * order_value

def hold_cost(order_value):
    """Rebuilt from real components instead of a flat multiplier."""
    return 20 + 60 + 0.05 * order_value

def three_tier_action(p, t1, t2):
    if p < t1:
        return "auto_approve"
    elif p < t2:
        return "flag_for_review"
    else:
        return "hold_for_verification"

t1_candidates = [0.16]
t2_candidates = np.arange(0.20, 0.90, 0.02)

results = []
for t1 in t1_candidates:
    for t2 in t2_candidates:
        if t2 <= t1:
            continue
        actions = pd.Series([three_tier_action(p, t1, t2) for p in probs])

        approve_mask = (actions == "auto_approve").values
        flag_mask = (actions == "flag_for_review").values
        hold_mask = (actions == "hold_for_verification").values

        missed_returns_mask = approve_mask & (y_true.values == 1)
        total_fn_cost = fn_cost(order_value[missed_returns_mask]).sum()
        total_flag_cost = flag_mask.sum() * FLAG_COST
        total_hold_cost = hold_cost(order_value[hold_mask]).sum()

        total_cost = total_fn_cost + total_flag_cost + total_hold_cost

        results.append({
            "t1": t1, "t2": round(t2, 2),
            "n_approved": approve_mask.sum(), "n_flagged": flag_mask.sum(), "n_held": hold_mask.sum(),
            "fn_cost": round(total_fn_cost, 2),
            "flag_cost": total_flag_cost,
            "hold_cost": round(total_hold_cost, 2),
            "total_cost": round(total_cost, 2),
        })

results_df = pd.DataFrame(results)
best = results_df.loc[results_df["total_cost"].idxmin()]

print("=== Three-tier cost search (component-based hold cost) - top 10 ===")
print(results_df.sort_values("total_cost").head(10).to_string(index=False))

print("\n=== Best (T1, T2) combination ===")
print(best.to_string())

results_df.to_csv("dSets/three_tier_cost_search.csv", index=False)
print("\nSaved: dSets/three_tier_cost_search.csv")

=== Three-tier cost search (component-based hold cost) - top 10 ===
  t1   t2  n_approved  n_flagged  n_held  fn_cost  flag_cost  hold_cost  total_cost
0.16 0.86         488        512       0   7685.2      20480       0.00    28165.20
0.16 0.88         488        512       0   7685.2      20480       0.00    28165.20
0.16 0.84         488        512       0   7685.2      20480       0.00    28165.20
0.16 0.82         488        510       2   7685.2      20400     200.54    28285.74
0.16 0.78         488        510       2   7685.2      20400     200.54    28285.74
0.16 0.80         488        510       2   7685.2      20400     200.54    28285.74
0.16 0.76         488        510       2   7685.2      20400     200.54    28285.74
0.16 0.74         488        509       3   7685.2      20360     329.81    28375.01
0.16 0.72         488        507       5   7685.2      20280     636.25    28601.45
0.16 0.70         488        503       9   7685.2      20120    1121.26    28926.45

=== Bes